In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from contextlib import AsyncExitStack
import json
from jsonschema import validate,ValidationError
from typing import Annotated, Sequence, TypedDict, Any, Dict,Union
from langchain_core.messages import BaseMessage,SystemMessage,AIMessage,ToolMessage,HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
# from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_core.tools import StructuredTool
from pydantic import create_model
import nest_asyncio 
from dotenv import load_dotenv
import os


In [ ]:
load_dotenv()

In [ ]:
def remove_descriptions(data, max_length=None):
    """
    Recursively remove description fields from JSON schema.
    If max_length is set, remove only descriptions longer than max_length.
    """
    if isinstance(data, dict):
        new_dict = {}
        for key, value in data.items():


            if key == "description":
                if max_length is None:  
                    continue
                elif isinstance(value, str) and len(value) > max_length:
                    continue  

            new_dict[key] = remove_descriptions(value, max_length)
        return new_dict

    elif isinstance(data, list):
        return [remove_descriptions(item, max_length) for item in data]

    return data


def validate_arguments(inputs_args, schema):
    try:
        validate(instance=inputs_args, schema=schema)
        print(f"---------Valid Arguments---------")
        return "Valid"
    except ValidationError as e:
        print(f"---------Invalid Arguments-------")
        return f"Invalid as {e}"

In [ ]:
from typing import List,Union
MAP = {
    "string":str,
    "number":float,
    "integer":int,
    "boolean":bool,
    "array":List[str],
    "object":dict
}


def json_to_model(name,schema):
    fields = {}
    required = schema.get("required",[])
    properties = schema.get("properties",{})


    for prop,rules in properties.items():
        if "anyOf" in rules:
            possible = []
            for option in rules["anyOf"]:
                if option["type"] == "string":
                    possible.append(MAP["string"])
                elif option["type"] == "array":
                    possible.append(MAP["array"])

            fields[prop] = (Union[tuple(possible)],... if prop in required else None)

        else:
            py_type = MAP[rules["type"]]
            fields[prop] = (py_type, ... if prop in required else None)

    return create_model(name, **fields)


In [ ]:
async def mcp_execute(session, tool_name: str, **kwargs):
    """Generic executor for ANY MCP tool."""
    result = await session.call_tool(tool_name, kwargs)
    return result

In [ ]:


def build_tool_from_schema(tool_name,tool_description,tool_schema,session):
    new_tool_name = tool_name.replace("-","_")
    new_tool_name = new_tool_name+"_Args"
    Arg_model = json_to_model(new_tool_name,tool_schema)

    async def wrapper(**kwargs):
        try:
            return await mcp_execute(
                session=session,
                tool_name=tool_name,
                **kwargs
            )
        except Exception as e:
            return f"TOOL ERROR: {type(e).__name__}: {str(e)}"


    nest_asyncio.apply()

    def sync_wrapper(**kwargs):
        import asyncio
        return asyncio.get_event_loop().run_until_complete(wrapper(**kwargs))

    tool = StructuredTool.from_function(
        name=tool_name,
        description=tool_description,
        func=sync_wrapper,
        args_schema=Arg_model
    )

    return tool

In [ ]:
def load_config() -> Union[Dict, None]:
    config_path = "mcp.json"

    try:
        with open(config_path) as f:
            config = json.load(f)

            mcp_servers = config.get("mcpServers", {})

            if not mcp_servers:
                print("No MCP servers found")

            return mcp_servers

    except Exception as e:
        print(f"Unable to open Config at path {config_path} as {e}")
        return None

async def configure_mcp(mcp_servers):


    input_schemas = {}
    name_to_tool = {}
    tools_list = []
    stack = AsyncExitStack()
    await stack.__aenter__()

    try:
        for server_name, server_info in mcp_servers.items():
            print(f"Connecting to server {server_name}...")

            server_param = StdioServerParameters(
                command=server_info["command"],
                args=server_info["args"],
                env=server_info.get("env")
            )

            read, write = await stack.enter_async_context(stdio_client(server_param))

            session = await stack.enter_async_context(
                ClientSession(read_stream=read, write_stream=write)
            )

            await session.initialize()
            print(f"Session initialized for {server_name}")



            server_tools = await session.list_tools()
            for tool in server_tools.tools:
                clean_schema = remove_descriptions(tool.inputSchema, max_length=200)  # or None
                input_schemas[tool.name] = clean_schema
                create_tool = build_tool_from_schema(tool.name,tool.description,clean_schema,session)
                name_to_tool[tool.name] = create_tool
                tools_list.append(create_tool)

        schema_path = "Input_schema.json"
        os.makedirs("schemas", exist_ok=True)
        with open("Input_schema.json", "w") as f:
            json.dump(input_schemas, f, indent=3)
            print(f"Schema Dumped at {schema_path}")

        return stack,input_schemas,tools_list,name_to_tool

    except Exception as e:
        print(f"Stack cloased due to some problem as {e}")
        
        await stack.aclose()

In [ ]:
def pretty_print_result(result):
    print("\n" + "="*60)
    print("FINAL AGENT OUTPUT")
    print("="*60)

    for msg in result["messages"]:
        if msg.__class__.__name__ == "HumanMessage":
            print("\n🧑 USER:")
            print(msg.content)

        elif msg.__class__.__name__ == "AIMessage":
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                print("\n🤖 ASSISTANT (Tool Call Request):")
                print(f"  → Tool: {msg.tool_calls[0]['name']}")
                print(f"  → Args: {msg.tool_calls[0]['args']}")
            else:
                print("\n🤖 ASSISTANT:")
                print(msg.content)

        elif msg.__class__.__name__ == "ToolMessage":
            print("\n🛠️ TOOL RESPONSE:")
            print(msg.content)

        else:
            print("\n❓ UNKNOWN MESSAGE TYPE:")
            print(msg)
    
    print("\n" + "="*60 + "\n")



In [ ]:

mcp_server = load_config()
stack, GLOBAL_SCHEMA, tools, GLOBAL_NAME_TO_TOOL = await configure_mcp(mcp_server)



llm = ChatOpenAI( 
    model="gpt-4o",
      openai_api_key=os.getenv("OPEN_AI_API_KEY"), 
      openai_api_base=os.getenv("OPEN_AI_API_BASE"),
    ).bind_tools(tools)


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    tool_calls = []


def model_call(state: AgentState):

    sys_prompt = SystemMessage(content="""
You are a precise cybersecurity agent with acess to differnet tools when executing any tool if you get wrong schema correct it
with the validation meassage and try again and always summarize the final output after the tool calls.
""")

    llm_input = [sys_prompt] + state["messages"]
    response: AIMessage = llm.invoke(llm_input)


    print(f"Reasoner: {response}")

    return {
        "messages": [response],
    }



def run_tool(state: AgentState):
    last = state["messages"][-1]

    tool_messages = []

    for tc in last.tool_calls:
        tool_id = tc["id"]
        tool_name = tc["name"]
        tool_args = tc["args"]

        validation = validate_arguments(tool_args, GLOBAL_SCHEMA[tool_name])
        if validation != "Valid":
            print(validation)
            tool_messages.append(
                ToolMessage(
                    content=validation,
                    name=tool_name,
                    tool_call_id=tool_id
                )
            )
        else:
            result = GLOBAL_NAME_TO_TOOL[tool_name].run(tool_args)
            tool_messages.append(
                ToolMessage(
                    content=result.content[0].text,
                    name=tool_name,
                    tool_call_id=tool_id
                )
            )


    return {
        "messages": tool_messages,
    }



def route_reasoner(state: AgentState):
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


graph = StateGraph(AgentState)

graph.add_node("reasoner", model_call)
graph.add_node("run_tool", run_tool)

graph.add_edge(START, "reasoner")


graph.add_conditional_edges(
    "reasoner",
    route_reasoner,
    {"tools": "run_tool", END: END},
)


graph.add_edge("run_tool", "reasoner")

react = graph.compile(debug=False)


conversation_history = []
user_input = input("Enter: ")
while user_input.lower() != "exit":
    conversation_history.append(HumanMessage(content=user_input))
    
    # Initialize tools_run on first invoke
    state_input = {
        "messages": conversation_history,
    }
    
    result = react.invoke(state_input)
    
    print(f"User: {user_input}")
    print(f"AI: {result['messages'][-1].content}")
    

    user_input = input("Enter: ")

await stack.aclose()



Connecting to server nmap...
Session initialized for nmap
Connecting to server sqlmap...
Session initialized for sqlmap
Connecting to server ffuf...
Session initialized for ffuf
Connecting to server masscan...
Session initialized for masscan
Connecting to server sslscan...
Session initialized for sslscan
Schema Dumped at Input_schema.json
Reasoner: content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 317, 'total_tokens': 348, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_b54fe76834', 'id': 'chatcmpl-CnhYhHhOrW6xWLJELUq86kk3HF0Fv', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019b2b82-bacd-7393-8632-1f05b4e2d43b-0' tool_calls=[{'name': 'do-sqlmap', 'args': 

In [ ]:
pretty_print_result(result)


FINAL AGENT OUTPUT

🧑 USER:
Can you check if my website https://testphp.vulnweb.com  is vulnerable?

🧑 USER:
so what can we do ?

🧑 USER:
do one by one

🤖 ASSISTANT (Tool Call Request):
  → Tool: do-nmap
  → Args: {'nmap_args': '-Pn', 'target': 'testphp.vulnweb.com'}

🛠️ TOOL RESPONSE:
Starting Nmap 7.98 ( https://nmap.org ) at 2025-12-17 14:25 +0530
Nmap scan report for testphp.vulnweb.com (44.228.249.3)
Host is up (0.28s latency).
rDNS record for 44.228.249.3: ec2-44-228-249-3.us-west-2.compute.amazonaws.com
Not shown: 999 filtered tcp ports (no-response)
PORT   STATE SERVICE
80/tcp open  http

Nmap done: 1 IP address (1 host up) scanned in 177.83 seconds

 nmap completed successfully

🤖 ASSISTANT (Tool Call Request):
  → Tool: do-sqlmap
  → Args: {'sqlmap_args': '-u', 'url': 'https://testphp.vulnweb.com'}

🛠️ TOOL RESPONSE:
sqlmap exited with code 2

🤖 ASSISTANT (Tool Call Request):
  → Tool: do-sqlmap
  → Args: {'sqlmap_args': ['-u'], 'url': 'https://testphp.vulnweb.com'}

🛠️ TOOL